2. 네이버 ( https://www.naver.com )에서 아래 예시와 같이 사용자에게 검색어와 저장할 파일들의
경로를 입력 받은 후 해당 키워드로 검색하여 나오는 결과에서 블로그만 선택한 후 15건의 데이
터를 아래 예시와 같이 xlsx, csv 형식으로 저장하세요.
(편의상 검색어는 “서진수 빅데이터”로 해보세요 ^^

In [ ]:
# naver.com 사이트에서 특정 키워드로 자동 검색하기 
#Step 1. 필요한 모듈을 로딩합니다 
from selenium import webdriver 
from selenium.webdriver.common.by import By 
from selenium.webdriver.common.keys import Keys 
from selenium.webdriver.chrome.service import Service 
import time           
#Step 2. 사용자에게 검색 관련 정보들을 입력 받습니다. 
print("=" *100) 
print(" 이 크롤러는 naver 사이트의 블로그 자료 수집용 웹크롤러입니다.") 
print("=" *100) 
query_txt = input('1.수집할 자료의 키워드는 무엇입니까?(예: 서진수 빅데이터): ') 
print("\n") 
#Step 3. 크롬 드라이버 설정 및 웹 페이지 열기 
s = Service("c:/py_temp/chromedriver.exe") 
driver = webdriver.Chrome(service=s) 
url = 'https://www.naver.com/' 
driver.get(url) 
time.sleep(5) 
driver.maximize_window() 
#Step 4. 자동으로 검색어 입력 후 조회하기 
element = driver.find_element(By.ID,'query') 
driver.find_element(By.ID,'query').click( ) 
element.send_keys(query_txt) 
element.send_keys("\n") 

#Step 5.블로그 선택하기
driver.find_element(By.LINK_TEXT,'블로그').click()
time.sleep(2)



#Step 6.Beautiful Soup 로 본문 내용만 추출하기
from bs4 import BeautifulSoup
html_1 = driver.page_source #현재 페이지의 전체 소스코드를 다 가져오기
soup_1 = BeautifulSoup(html_1, 'html.parser')

content_1 = soup_1.find('div','sds-comps-vertical-layout sds-comps-full-layout fds-news-item-list-tab').find_all('div','sds-comps-vertical-layout sds-comps-full-layout YWTMk0ahJUsxq4uCx9gX')
for i in content_1 :
    print(i.get_text().replace("\n"," ").strip())
    print("\n")



# Step 9. 데이터 저장용 리스트 생성














# Step 10. DataFrame 생성 및 저장


# 엑셀 저장
df.to_excel(fx_name, index=False, engine='openpyxl')

# CSV 저장
df.to_csv(fc_name, index=False, encoding="utf-8-sig")

print('요청하신 데이터 수집 작업이 정상적으로 완료되었습니다')
driver.quit()








In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
import time
import pandas as pd

s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

query_txt = input("검색어 입력: ")

driver.get("https://www.naver.com/")
time.sleep(2)

driver.find_element(By.ID,"query").send_keys(query_txt + "\n")
time.sleep(2)

driver.find_element(By.LINK_TEXT,"블로그").click()
time.sleep(2)

# ✅ 무한스크롤
prev_height = driver.execute_script("return document.body.scrollHeight")

while True:
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)
    curr_height = driver.execute_script("return document.body.scrollHeight")
    if curr_height == prev_height:
        break
    prev_height = curr_height

# ✅ 데이터 수집
items = driver.find_elements(By.CSS_SELECTOR,"div.total_wrap")

no2=[]
title2=[]
summary2=[]
date2=[]
nick2=[]
url2=[]

for idx, item in enumerate(items, start=1):
    try:
        title = item.find_element(By.CSS_SELECTOR,"a.title_link").text
        url = item.find_element(By.CSS_SELECTOR,"a.title_link").get_attribute("href")
        summary = item.find_element(By.CSS_SELECTOR,"div.dsc_wrap").text
        date = item.find_element(By.CSS_SELECTOR,"span.sub").text
        nick = item.find_element(By.CSS_SELECTOR,"a.name").text

        no2.append(idx)
        title2.append(title)
        summary2.append(summary)
        date2.append(date)
        nick2.append(nick)
        url2.append(url)

    except:
        continue

# 데이터프레임
df = pd.DataFrame({
    "번호":no2,
    "제목":title2,
    "요약내용":summary2,
    "작성일자":date2,
    "블로그닉네임":nick2,
    "URL":url2
})

df.to_csv("naver_blog.csv",index=False,encoding="utf-8-sig")

print("수집 완료")
driver.quit()

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
import pandas as pd
import time
import os

# Step 1. 필요한 모듈 로딩 및 파일 경로 설정
print("=" * 100)
print(" 이 크롤러는 naver 사이트의 블로그 자료 수집용 웹크롤러입니다.")
print("=" * 100)
query_txt = input('1.수집할 자료의 키워드는 무엇입니까?(예: 서진수 빅데이터): ')
fc_name = input('2.csv로 저장할 파일명을 입력하세요(예: result.csv): ')
fx_name = input('3.xls로 저장할 파일명을 입력하세요(예: result.xlsx): ')

# Step 2. 크롬 드라이버 설정 및 웹 페이지 열기
# 경로가 다르다면 본인의 chromedriver 위치로 수정하세요.
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)
url = 'https://www.naver.com/'
driver.get(url)
time.sleep(2)
driver.maximize_window()

# Step 3. 자동으로 검색어 입력 후 조회하기
element = driver.find_element(By.ID, 'query')
element.send_keys(query_txt)
element.send_keys("\n")
time.sleep(2)

# Step 4. 블로그 탭 선택하기
driver.find_element(By.LINK_TEXT, '블로그').click()
time.sleep(3)

# Step 5. 무한 스크롤 처리 (데이터를 충분히 확보하기 위해 3번 스크롤)
for _ in range(3):
    driver.find_element(By.TAG_NAME, 'body').send_keys(Keys.END)
    time.sleep(1.5)

# Step 6. BeautifulSoup으로 본문 내용 추출
html_1 = driver.page_source
soup_1 = BeautifulSoup(html_1, 'html.parser')

# 각 블로그 포스트를 감싸는 아이템 리스트 추출
# 네이버 블로그 검색 결과의 개별 아이템 클래스는 보통 'bx' 혹은 'view_wrap' 단위를 포함합니다.
content_list = soup_1.select('li.bx')

# 데이터 저장용 리스트 생성
no = []
title_list = []
summary_list = []
date_list = []
nickname_list = []

print("\n데이터 수집을 시작합니다...\n")

# Step 7. 루프를 돌며 상세 데이터 추출
for idx, cont in enumerate(content_list, 1):
    try:
        # 제목 추출 (제공해주신 클래스 기반)
        title_tag = cont.find('span', class_='sds-comps-text-type-headline1')
        title = title_tag.get_text(strip=True) if title_tag else "제목 없음"
        
        # 요약내용 추출
        summary_tag = cont.find('span', class_='sds-comps-text-type-body1')
        summary = summary_tag.get_text(strip=True) if summary_tag else "내용 없음"
        
        # 작성일자 추출
        date_tag = cont.find('span', class_='sds-comps-profile-info-subtext')
        date = date_tag.get_text(strip=True) if date_tag else "날짜 없음"
        
        # 블로그 닉네임 추출 (프로필 영역)
        nick_tag = cont.select_one('.sds-comps-profile-info-title-text')
        nickname = nick_tag.get_text(strip=True) if nick_tag else "닉네임 없음"

        # 리스트에 추가
        no.append(idx)
        title_list.append(title)
        summary_list.append(summary)
        date_list.append(date)
        nickname_list.append(nickname)

        # 콘솔 출력 (진행 확인용)
        print(f"[{idx}] {title}")

    except Exception as e:
        continue

# Step 8. DataFrame 생성 및 저장
df = pd.DataFrame()
df['번호'] = no
df['제목'] = title_list
df['요약내용'] = summary_list
df['작성일자'] = date_list
df['블로그닉네임'] = nickname_list

# 엑셀 및 CSV 저장 (사용자 입력 파일명 사용)
df.to_excel(fx_name, index=False, engine='openpyxl')
df.to_csv(fc_name, index=False, encoding="utf-8-sig")

print("\n" + "="*50)
print(f"요청하신 데이터 수집 작업이 완료되었습니다.")
print(f"저장된 파일: {fx_name}, {fc_name}")
print("="*50)

driver.quit()